# Proyek Analisis Data: Bike Sharing Dataset
- **Nama:** Muh. Arif
- **Email:** arif80352@gmail.com
- **ID Dicoding:** arifakasah

## Menentukan Pertanyaan Bisnis

- **Pertanyaan 1:** Bagaimana pengaruh kondisi cuaca (weathersit) terhadap rata-rata jumlah penyewaan sepeda harian di Washington D.C. selama periode 2011-2012, dan kondisi cuaca mana yang menyebabkan penurunan penyewaan tertinggi?

- **Pertanyaan 2:** Pada jam berapa terjadi puncak penyewaan sepeda oleh pengguna registered dibandingkan casual pada hari kerja di tahun 2012, dan bagaimana perbedaan pola tersebut dapat digunakan untuk mengoptimalkan distribusi sepeda?

## Import Semua Packages/Library yang Digunakan

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## Data Wrangling

### Gathering Data

#### Load Dataset day.csv dan hour.csv

In [ ]:
day_df = pd.read_csv('../day.csv')
hour_df = pd.read_csv('../hour.csv')

print('day.csv shape:', day_df.shape)
print('hour.csv shape:', hour_df.shape)

In [ ]:
day_df.head()

In [ ]:
hour_df.head()

**Insight:**
- Dataset day.csv berisi 731 baris dan 16 kolom (data harian)
- Dataset hour.csv berisi 17.379 baris dan 17 kolom (data per jam, tambahan kolom `hr`)
- Kedua dataset memiliki kolom yang sama kecuali `hr` yang hanya ada di hour.csv

### Assessing Data

#### Identifying Missing Values and Data Quality Problems

In [ ]:
# Cek info tipe data
print('Info day.csv:')
day_df.info()
print('\n' + '='*50 + '\n')
print('Info hour.csv:')
hour_df.info()

In [ ]:
# Cek missing values
print('Missing values pada day.csv:')
print(day_df.isnull().sum())
print('\nMissing values pada hour.csv:')
print(hour_df.isnull().sum())

In [ ]:
# Cek duplicate data
print('Duplikat pada day.csv:', day_df.duplicated().sum())
print('Duplikat pada hour.csv:', hour_df.duplicated().sum())

In [ ]:
# Cek statistik deskriptif
day_df.describe()

In [ ]:
hour_df.describe()

In [ ]:
# Cek invalid values pada kolom kategorikal
print('Nilai unik weathersit day:', sorted(day_df['weathersit'].unique()))
print('Nilai unik weathersit hour:', sorted(hour_df['weathersit'].unique()))
print('Nilai unik season day:', sorted(day_df['season'].unique()))
print('Nilai unik season hour:', sorted(hour_df['season'].unique()))
print('Nilai unik yr:', sorted(day_df['yr'].unique()))

In [ ]:
# Cek outlier menggunakan IQR
def detect_outliers_iqr(df, columns):
    outlier_info = {}
    for col in columns:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)][col]
        outlier_info[col] = len(outliers)
    return outlier_info

numeric_cols = ['temp', 'atemp', 'hum', 'windspeed', 'casual', 'registered', 'cnt']
print('Outlier pada day.csv:')
for col, count in detect_outliers_iqr(day_df, numeric_cols).items():
    print(f'  {col}: {count} outlier')

print('\nOutlier pada hour.csv:')
for col, count in detect_outliers_iqr(hour_df, numeric_cols).items():
    print(f'  {col}: {count} outlier')

In [ ]:
# Cek inconsistent value: hum = 0 (kelembaban 0% secara fisik tidak mungkin)
print('Jumlah hum = 0 pada hour.csv:', (hour_df['hum'] == 0).sum())
print('Jumlah hum = 0 pada day.csv:', (day_df['hum'] == 0).sum())

**Steps to Take:**
- Tidak ada missing values pada kedua dataset, sehingga tidak perlu handling missing values
- Tidak ada data duplikat pada kedua dataset
- Terdapat outlier pada kolom `windspeed`, `casual`, dan `cnt` - outlier pada `casual` dan `cnt` kemungkinan data valid (hari dengan penyewaan tinggi)
- Terdapat nilai `hum = 0` pada hour.csv (4 baris) yang secara fisik tidak mungkin - perlu diganti dengan median
- Kolom `dteday` perlu dikonversi ke tipe datetime
- Kolom kategorikal (`season`, `yr`, `mnth`, `hr`, `holiday`, `weekday`, `workingday`, `weathersit`) perlu dikonversi ke tipe category

**Insight:**
- Dataset cukup bersih dengan tidak adanya missing values dan duplikat
- Masalah utama yang ditemukan: nilai hum=0 yang tidak realistis dan tipe data yang belum optimal

### Cleaning Data

#### Fixing Data Type and Inconsistent Values

In [ ]:
# Konversi dteday ke datetime
day_df['dteday'] = pd.to_datetime(day_df['dteday'])
hour_df['dteday'] = pd.to_datetime(hour_df['dteday'])

# Konversi kolom kategori
category_cols = ['season', 'yr', 'mnth', 'holiday', 'weekday', 'workingday', 'weathersit']
for col in category_cols:
    day_df[col] = day_df[col].astype('category')
    hour_df[col] = hour_df[col].astype('category')

hour_df['hr'] = hour_df['hr'].astype('category')

print('Tipe data setelah konversi:')
print(day_df.dtypes)

In [ ]:
# Menangani hum = 0 pada hour_df (replace dengan median)
zero_hum_count = (hour_df['hum'] == 0).sum()
print(f'Jumlah hum = 0 pada hour.csv: {zero_hum_count}')

median_hum = hour_df[hour_df['hum'] > 0]['hum'].median()
hour_df['hum'] = hour_df['hum'].replace(0, median_hum)
print(f'Median hum yang digunakan: {median_hum}')
print(f'Jumlah hum = 0 setelah cleaning: {(hour_df["hum"] == 0).sum()}')

In [ ]:
# Mapping label untuk visualisasi
season_map = {1: 'Spring', 2: 'Summer', 3: 'Fall', 4: 'Winter'}
weathersit_map = {
    1: 'Clear',
    2: 'Mist/Cloudy',
    3: 'Light Snow/Rain',
    4: 'Heavy Rain/Snow'
}
yr_map = {0: '2011', 1: '2012'}

day_df['season_label'] = day_df['season'].map(season_map)
day_df['weathersit_label'] = day_df['weathersit'].map(weathersit_map)
day_df['yr_label'] = day_df['yr'].map(yr_map)

hour_df['season_label'] = hour_df['season'].map(season_map)
hour_df['weathersit_label'] = hour_df['weathersit'].map(weathersit_map)
hour_df['yr_label'] = hour_df['yr'].map(yr_map)

day_df.head()

**Insight:**
- Konversi tipe data berhasil dilakukan
- Nilai hum=0 telah diganti dengan median (0.63)
- Label mapping ditambahkan untuk memudahkan interpretasi visualisasi

## Exploratory Data Analysis (EDA)

### Explore Pengaruh Kondisi Cuaca terhadap Penyewaan (Pertanyaan 1)

In [ ]:
# Rata-rata penyewaan berdasarkan kondisi cuaca
weather_stats = day_df.groupby('weathersit_label', observed=True)['cnt'].agg(['mean', 'median', 'std', 'count'])
weather_stats.columns = ['Rata-rata', 'Median', 'Std Dev', 'Jumlah Hari']
weather_stats = weather_stats.sort_values('Rata-rata', ascending=False)
print('Statistik Penyewaan Berdasarkan Kondisi Cuaca:')
print(weather_stats.round(2))

In [ ]:
# Rata-rata penyewaan berdasarkan musim
season_stats = day_df.groupby('season_label', observed=True)['cnt'].agg(['mean', 'median', 'std', 'count'])
season_stats.columns = ['Rata-rata', 'Median', 'Std Dev', 'Jumlah Hari']
print('Statistik Penyewaan Berdasarkan Musim:')
print(season_stats.round(2))

In [ ]:
# Rata-rata penyewaan berdasarkan cuaca dan tahun
weather_yr = day_df.groupby(['weathersit_label', 'yr_label'], observed=True)['cnt'].mean().unstack()
print('Rata-rata Penyewaan per Kondisi Cuaca dan Tahun:')
print(weather_yr.round(2))

In [ ]:
# Korelasi antara variabel cuaca dan penyewaan
weather_corr = day_df[['temp', 'atemp', 'hum', 'windspeed', 'cnt']].corr()
print('Korelasi Variabel Cuaca dengan Jumlah Penyewaan:')
print(weather_corr['cnt'].sort_values(ascending=False).round(4))

### Explore Pola Penyewaan per Jam (Pertanyaan 2)

In [ ]:
# Filter data tahun 2012 dan hari kerja
hour_2012_workday = hour_df[(hour_df['yr'] == 1) & (hour_df['workingday'] == 1)]
print(f'Jumlah data tahun 2012 hari kerja: {len(hour_2012_workday)} baris')

# Rata-rata penyewaan per jam
hourly_stats = hour_2012_workday.groupby('hr', observed=True)[['casual', 'registered', 'cnt']].mean()
print('\nRata-rata Penyewaan per Jam (2012, Hari Kerja):')
print(hourly_stats.round(2))

In [ ]:
# Jam puncak untuk registered dan casual
peak_registered = hourly_stats['registered'].idxmax()
peak_registered_val = hourly_stats['registered'].max()
print(f'Jam puncak penyewaan Registered: Jam {peak_registered}:00 dengan rata-rata {peak_registered_val:.0f} sepeda')

peak_casual = hourly_stats['casual'].idxmax()
peak_casual_val = hourly_stats['casual'].max()
print(f'Jam puncak penyewaan Casual: Jam {peak_casual}:00 dengan rata-rata {peak_casual_val:.0f} sepeda')

# Jam terendah
min_registered = hourly_stats['registered'].idxmin()
min_casual = hourly_stats['casual'].idxmin()
print(f'\nJam terendah Registered: Jam {min_registered}:00')
print(f'Jam terendah Casual: Jam {min_casual}:00')

In [ ]:
# Perbandingan hari kerja vs weekend
hourly_workday = hour_df[(hour_df['yr'] == 1) & (hour_df['workingday'] == 1)].groupby('hr', observed=True)['cnt'].mean()
hourly_weekend = hour_df[(hour_df['yr'] == 1) & (hour_df['workingday'] == 0)].groupby('hr', observed=True)['cnt'].mean()

print('Rata-rata Penyewaan per Jam - Hari Kerja vs Weekend (2012):')
comparison = pd.DataFrame({'Hari Kerja': hourly_workday, 'Weekend': hourly_weekend})
print(comparison.round(2))

In [ ]:
# Persentase casual vs registered
total_casual = day_df['casual'].sum()
total_registered = day_df['registered'].sum()
total_all = day_df['cnt'].sum()

print(f'Total Casual: {total_casual:,} ({total_casual/total_all*100:.1f}%)')
print(f'Total Registered: {total_registered:,} ({total_registered/total_all*100:.1f}%)')
print(f'Total Keseluruhan: {total_all:,}')

**Insight:**
- Cuaca Clear menghasilkan rata-rata penyewaan tertinggi, sedangkan Light Snow/Rain terendah
- Suhu memiliki korelasi positif kuat (0.627) dengan penyewaan
- Registered users memiliki pola bimodal (jam 8 dan 17-18), casual users unimodal (jam 17)
- Registered users mendominasi 81.2% total penyewaan

## Visualization & Explanatory Analysis

### Pertanyaan 1: Pengaruh Kondisi Cuaca terhadap Penyewaan

In [ ]:
# Visualisasi 1: Bar chart rata-rata penyewaan berdasarkan kondisi cuaca
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart cuaca
weather_order = ['Clear', 'Mist/Cloudy', 'Light Snow/Rain']
colors_weather = ['#2ecc71', '#f39c12', '#e74c3c']

weather_avg = day_df.groupby('weathersit_label', observed=True)['cnt'].mean()
weather_avg = weather_avg.reindex(weather_order)

bars = axes[0].bar(weather_avg.index, weather_avg.values, color=colors_weather, edgecolor='black', linewidth=0.5)
axes[0].set_title('Rata-rata Penyewaan Sepeda Harian\nBerdasarkan Kondisi Cuaca (2011-2012)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Kondisi Cuaca', fontsize=11)
axes[0].set_ylabel('Rata-rata Jumlah Penyewaan', fontsize=11)

for bar, val in zip(bars, weather_avg.values):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 50,
                 f'{val:.0f}', ha='center', va='bottom', fontweight='bold', fontsize=11)

# Bar chart cuaca per tahun
weather_yr_plot = day_df.groupby(['weathersit_label', 'yr_label'], observed=True)['cnt'].mean().unstack()
weather_yr_plot = weather_yr_plot.reindex(weather_order)

x = np.arange(len(weather_order))
width = 0.35
bars1 = axes[1].bar(x - width/2, weather_yr_plot['2011'], width, label='2011', color='#3498db', edgecolor='black', linewidth=0.5)
bars2 = axes[1].bar(x + width/2, weather_yr_plot['2012'], width, label='2012', color='#e67e22', edgecolor='black', linewidth=0.5)

axes[1].set_title('Rata-rata Penyewaan Berdasarkan Cuaca\nPer Tahun (2011 vs 2012)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Kondisi Cuaca', fontsize=11)
axes[1].set_ylabel('Rata-rata Jumlah Penyewaan', fontsize=11)
axes[1].set_xticks(x)
axes[1].set_xticklabels(weather_order)
axes[1].legend()

for bar in bars1:
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 30,
                 f'{bar.get_height():.0f}', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 30,
                 f'{bar.get_height():.0f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Heatmap korelasi variabel cuaca
fig, ax = plt.subplots(figsize=(8, 6))
corr_data = day_df[['temp', 'atemp', 'hum', 'windspeed', 'cnt']].corr()
sns.heatmap(corr_data, annot=True, cmap='RdYlGn', center=0, fmt='.3f',
            linewidths=0.5, ax=ax, vmin=-1, vmax=1)
ax.set_title('Heatmap Korelasi Variabel Cuaca dengan\nJumlah Penyewaan Sepeda', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### Pertanyaan 2: Pola Penyewaan per Jam (Registered vs Casual)

In [ ]:
# Visualisasi 2: Line chart pola penyewaan per jam
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Line chart registered vs casual pada hari kerja 2012
hours = hourly_stats.index.astype(int)

axes[0].plot(hours, hourly_stats['registered'], marker='o', linewidth=2.5,
             color='#2980b9', label='Registered', markersize=5)
axes[0].plot(hours, hourly_stats['casual'], marker='s', linewidth=2.5,
             color='#e74c3c', label='Casual', markersize=5)
axes[0].fill_between(hours, hourly_stats['registered'], alpha=0.1, color='#2980b9')
axes[0].fill_between(hours, hourly_stats['casual'], alpha=0.1, color='#e74c3c')

# Anotasi jam puncak
axes[0].axvline(x=peak_registered, color='#2980b9', linestyle='--', alpha=0.5)
axes[0].axvline(x=peak_casual, color='#e74c3c', linestyle='--', alpha=0.5)
axes[0].annotate(f'Peak Registered\nJam {peak_registered}:00',
                 xy=(peak_registered, peak_registered_val),
                 xytext=(peak_registered-3, peak_registered_val+20),
                 arrowprops=dict(arrowstyle='->', color='#2980b9'),
                 fontsize=9, color='#2980b9', fontweight='bold')
axes[0].annotate(f'Peak Casual\nJam {peak_casual}:00',
                 xy=(peak_casual, peak_casual_val),
                 xytext=(peak_casual+2, peak_casual_val+15),
                 arrowprops=dict(arrowstyle='->', color='#e74c3c'),
                 fontsize=9, color='#e74c3c', fontweight='bold')

axes[0].set_title('Pola Penyewaan Sepeda per Jam\n(Registered vs Casual) - Hari Kerja 2012',
                  fontsize=13, fontweight='bold')
axes[0].set_xlabel('Jam', fontsize=11)
axes[0].set_ylabel('Rata-rata Jumlah Penyewaan', fontsize=11)
axes[0].set_xticks(range(0, 24))
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Line chart hari kerja vs weekend
axes[1].plot(hours, hourly_workday.values, marker='o', linewidth=2.5,
             color='#27ae60', label='Hari Kerja', markersize=5)
axes[1].plot(hours, hourly_weekend.values, marker='s', linewidth=2.5,
             color='#8e44ad', label='Weekend', markersize=5)
axes[1].fill_between(hours, hourly_workday.values, alpha=0.1, color='#27ae60')
axes[1].fill_between(hours, hourly_weekend.values, alpha=0.1, color='#8e44ad')

axes[1].set_title('Pola Penyewaan Sepeda per Jam\n(Hari Kerja vs Weekend) - 2012',
                  fontsize=13, fontweight='bold')
axes[1].set_xlabel('Jam', fontsize=11)
axes[1].set_ylabel('Rata-rata Jumlah Penyewaan', fontsize=11)
axes[1].set_xticks(range(0, 24))
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Pie chart casual vs registered
fig, ax = plt.subplots(figsize=(8, 6))
sizes = [total_casual, total_registered]
labels = [f'Casual\n({total_casual/total_all*100:.1f}%)', f'Registered\n({total_registered/total_all*100:.1f}%)']
colors = ['#e74c3c', '#2980b9']
explode = (0.05, 0.05)

wedges, texts, autotexts = ax.pie(sizes, explode=explode, labels=labels, colors=colors,
                                   autopct='%1.1f%%', shadow=True, startangle=90,
                                   textprops={'fontsize': 12})
for autotext in autotexts:
    autotext.set_fontweight('bold')

ax.set_title('Proporsi Penyewaan Sepeda\n(Casual vs Registered) 2011-2012',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**Insight:**
- Visualisasi 1 menunjukkan penurunan signifikan penyewaan saat cuaca buruk
- Visualisasi 2 menunjukkan pola bimodal registered users dan unimodal casual users

## Analisis Lanjutan (Opsional)

In [ ]:
# Analisis Clustering berdasarkan jam dan tipe pengguna
hourly_full = hour_df.groupby('hr', observed=True)[['casual', 'registered']].mean()

# Kategorisasi jam berdasarkan volume penyewaan
hourly_full['total'] = hourly_full['casual'] + hourly_full['registered']
hourly_full['category'] = pd.cut(hourly_full['total'],
                                  bins=[0, 100, 300, float('inf')],
                                  labels=['Low', 'Medium', 'High'])

print('Kategori Volume Penyewaan per Jam:')
print(hourly_full[['total', 'category']])

**Insight:**
- Jam 0-5 termasuk kategori Low volume
- Jam 6-9 dan 16-21 termasuk Medium-High volume
- Puncak tertinggi pada jam 17:00 dengan rata-rata > 400 sepeda

## Conclusion & Recommendation

- **Conclusion pertanyaan 1:** Kondisi cuaca Clear (Cerah) menghasilkan rata-rata penyewaan tertinggi (~4.876 sepeda/hari), diikuti Mist/Cloudy (~4.034), dan Light Snow/Rain (~1.803). Terjadi penurunan signifikan sekitar 62.9% dari kondisi Clear ke Light Snow/Rain. Suhu memiliki korelasi positif kuat (r=0.627) dengan penyewaan. Pertumbuhan dari 2011 ke 2012 terjadi di semua kondisi cuaca.

- **Conclusion pertanyaan 2:** Registered users memiliki pola bimodal dengan puncak pada jam 8:00 (~479 sepeda) dan 17:00-18:00 (~526 sepeda), sesuai jam kerja. Casual users memiliki pola unimodal dengan puncak jam 17:00 (~74 sepeda). Pada weekend, puncak terjadi jam 12:00-15:00. Registered users mendominasi 81.2% total penyewaan.

**Rekomendasi Action Item:**

1. **Optimasi Distribusi Sepeda Berdasarkan Jam Puncak:**
   - Tambah stok sepeda di stasiun utama pada jam 7:00-9:00 dan 16:00-18:00 di hari kerja
   - Pada weekend, fokus distribusi jam 10:00-16:00 di area rekreasi

2. **Strategi Cuaca:**
   - Berikan promosi diskon saat cuaca hujan/salju untuk menjaga volume
   - Sediakan aksesori pelindung hujan di stasiun saat prediksi cuaca buruk

3. **Konversi Casual ke Registered:**
   - Targetkan promosi pada jam 12:00-16:00 di weekend ketika casual users aktif
   - Buat program loyalitas untuk meningkatkan retensi pengguna casual